# Recommender Evaluation Workflow

This notebook is an offline benchmark for the profile recommender. It uses a temporal holdout: train from older viewing history, hold out the most recent period, generate recommendations, and compare ranked results against held-out title and metadata signals.

The goal is to compare recommender changes against simple baselines before promoting new scoring weights or candidate sources.

## What This Measures

- exact title recovery: `hit_rate@k`, `recall@k`
- metadata fit: genre, language, and country coverage
- ranking quality at `k = 5, 10, 20`
- diversity across genre, language, country, media type, and decade
- novelty using inverse popularity
- source attribution for candidate generation
- diagnostics for low-performing profiles
- comparison against simple baselines

Exact title hits may be low because Netflix title parsing, TMDB IDs, and catalog availability are imperfect. Metadata-fit metrics are therefore important.

In [ ]:
import json
import math
import os
import random
import sys
from collections import Counter
from datetime import timedelta
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mysite.settings")
os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")

import django
django.setup()

from django.db.models import Max

from api.models import ExternalCatalogTitle, NetflixProfile, RecommendationFeedback, ViewingEvent
from api.services import recommendations as recs

In [ ]:
K_VALUES = [5, 10, 20]
HOLDOUT_DAYS = 90
RANDOM_SEED = 42
EXPORT_DIR = ROOT / "notebooks" / "results"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)

EXPERIMENTS = [
    {"name": "current", "weights": {}},
    {"name": "genre_heavy", "weights": {"content_similarity": 0.26, "genre_affinity": 0.30}},
    {"name": "quality_source_heavy", "weights": {"quality_confidence": 0.14, "source_strength": 0.10}},
    {"name": "language_country_heavy", "weights": {"language_affinity": 0.16, "country_affinity": 0.14}},
]

## Data Split Helpers

In [ ]:
def profile_windows(profile, holdout_days=HOLDOUT_DAYS):
    latest = ViewingEvent.objects.filter(profile=profile).aggregate(latest=Max("started_at"))["latest"]
    if latest is None:
        return [], []
    holdout_start = latest.date() - timedelta(days=holdout_days - 1)
    train_events = list(
        ViewingEvent.objects.filter(
            profile=profile,
            title__isnull=False,
            started_at__date__lt=holdout_start,
        ).select_related("title")
    )
    holdout_events = list(
        ViewingEvent.objects.filter(
            profile=profile,
            title__isnull=False,
            started_at__date__gte=holdout_start,
        ).select_related("title")
    )
    return train_events, holdout_events

def title_key(title):
    media_type = recs._media_type_for_title(title)
    if title.tmdb_id and media_type:
        return (media_type, str(title.tmdb_id))
    return ("name", (title.canonical_name or title.name).strip().casefold())

def catalog_key(candidate):
    return (candidate.media_type, str(candidate.external_id))

def weighted_rows_from_events(events, period_end):
    by_title = {}
    for event in events:
        row = by_title.setdefault(
            event.title_id,
            {"title": event.title, "weight": 0.0, "seconds": 0, "latest": event.started_at},
        )
        age_days = max(0, (period_end - event.started_at.date()).days)
        recency = recs.exp(-recs.log(2) * age_days / 45)
        row["seconds"] += event.duration_seconds
        row["weight"] += max(event.duration_seconds, 60) * recency
        row["latest"] = max(row["latest"], event.started_at)
    return sorted(by_title.values(), key=lambda row: row["weight"], reverse=True)

## Metric Helpers

In [ ]:
def values_from_events(events, extractor):
    values = set()
    for event in events:
        extracted = extractor(event.title)
        if not extracted:
            continue
        if isinstance(extracted, list):
            values.update(value for value in extracted if value)
        else:
            values.add(extracted)
    return values

def values_from_candidates(candidates, extractor):
    values = set()
    for candidate in candidates:
        extracted = extractor(candidate)
        if not extracted:
            continue
        if isinstance(extracted, list):
            values.update(value for value in extracted if value)
        else:
            values.add(extracted)
    return values

def coverage(heldout_values, recommended_values):
    if not heldout_values:
        return None
    return round(len(heldout_values & recommended_values) / len(heldout_values), 4)

def diversity(candidates):
    if not candidates:
        return 0
    dimensions = [
        [candidate.genres[0] if candidate.genres else "Unknown" for candidate in candidates],
        [candidate.original_language or "Unknown" for candidate in candidates],
        [candidate.origin_countries[0] if candidate.origin_countries else "Unknown" for candidate in candidates],
        [candidate.media_type for candidate in candidates],
        [recs._release_decade(candidate.release_year) or "Unknown" for candidate in candidates],
    ]
    scores = [len(set(values)) / max(1, len(values)) for values in dimensions]
    return round(sum(scores) / len(scores), 4)

def novelty(candidates):
    if not candidates:
        return 0
    scores = [1 / math.log2((candidate.popularity or 0) + 2) for candidate in candidates]
    return round(sum(scores) / len(scores), 4)

def source_counts(candidates):
    counts = Counter()
    for candidate in candidates:
        sources = candidate.metadata.get("candidate_sources", []) or ["cached"]
        for source in sources:
            counts[source] += 1
    return dict(counts)

def metric_row(profile_name, model_name, k, ranked_candidates, holdout_events, extra=None):
    top_k = ranked_candidates[:k]
    heldout_keys = {title_key(event.title) for event in holdout_events}
    recommended_keys = {catalog_key(candidate) for candidate in top_k}
    hits = heldout_keys & recommended_keys
    heldout_genres = values_from_events(holdout_events, lambda title: title.genres)
    heldout_languages = values_from_events(holdout_events, lambda title: title.original_language)
    heldout_countries = values_from_events(holdout_events, lambda title: title.origin_countries)
    recommended_genres = values_from_candidates(top_k, lambda candidate: candidate.genres)
    recommended_languages = values_from_candidates(top_k, lambda candidate: candidate.original_language)
    recommended_countries = values_from_candidates(top_k, lambda candidate: candidate.origin_countries)
    row = {
        "profile": profile_name,
        "model": model_name,
        "k": k,
        "hit_rate": 1.0 if hits else 0.0,
        "recall": round(len(hits) / max(1, len(heldout_keys)), 4),
        "genre_coverage": coverage(heldout_genres, recommended_genres),
        "language_overlap": coverage(heldout_languages, recommended_languages),
        "country_overlap": coverage(heldout_countries, recommended_countries),
        "diversity": diversity(top_k),
        "novelty": novelty(top_k),
        "source_counts": source_counts(top_k),
    }
    if extra:
        row.update(extra)
    return row

## Baselines

These simple recommenders make the benchmark useful. The production recommender should beat them on metadata-fit metrics, not only on exact title recovery.

In [ ]:
def watched_catalog_keys(weighted_titles):
    return {
        (recs._media_type_for_title(row["title"]), str(row["title"].tmdb_id))
        for row in weighted_titles
        if row["title"].tmdb_id and recs._media_type_for_title(row["title"])
    }

def eligible_catalog(weighted_titles):
    watched = watched_catalog_keys(weighted_titles)
    watched_names = {
        (row["title"].canonical_name or row["title"].name).strip().casefold()
        for row in weighted_titles
    }
    return [
        candidate
        for candidate in ExternalCatalogTitle.objects.filter(source="tmdb")
        if catalog_key(candidate) not in watched
        and candidate.title.strip().casefold() not in watched_names
        and (candidate.genres or candidate.overview)
    ]

def popular_baseline(weighted_titles, preferences, limit=20):
    return sorted(eligible_catalog(weighted_titles), key=lambda candidate: candidate.popularity or 0, reverse=True)[:limit]

def random_baseline(weighted_titles, preferences, limit=20):
    candidates = eligible_catalog(weighted_titles)
    random.shuffle(candidates)
    return candidates[:limit]

def genre_baseline(weighted_titles, preferences, limit=20):
    preferred = {row["value"] for row in preferences["genres"][:3]}
    candidates = eligible_catalog(weighted_titles)
    return sorted(
        candidates,
        key=lambda candidate: (
            len(preferred & set(candidate.genres or [])),
            candidate.popularity or 0,
        ),
        reverse=True,
    )[:limit]

def language_country_baseline(weighted_titles, preferences, limit=20):
    languages = {row["value"] for row in preferences["languages"][:2]}
    countries = {row["value"] for row in preferences["countries"][:3]}
    candidates = eligible_catalog(weighted_titles)
    return sorted(
        candidates,
        key=lambda candidate: (
            int(candidate.original_language in languages),
            len(countries & set(candidate.origin_countries or [])),
            candidate.popularity or 0,
        ),
        reverse=True,
    )[:limit]

BASELINES = {
    "popular_baseline": popular_baseline,
    "random_baseline": random_baseline,
    "genre_baseline": genre_baseline,
    "language_country_baseline": language_country_baseline,
}

## Production Recommender Experiments

In [ ]:
def normalize_weights(weights):
    total = sum(weights.values()) or 1
    return {key: round(value / total, 5) for key, value in weights.items()}

def apply_weight_override(hyperparameters, override):
    next_params = {**hyperparameters, "score_weights": dict(hyperparameters["score_weights"])}
    next_params["score_weights"].update(override)
    next_params["score_weights"] = normalize_weights(next_params["score_weights"])
    return next_params

def production_ranked(profile, weighted_titles, preferences, experiment):
    hyperparameters = recs._generate_hyperparameters(weighted_titles)
    hyperparameters = apply_weight_override(hyperparameters, experiment.get("weights", {}))
    feedback_actions = recs._feedback_map(profile)
    candidates = recs._discover_candidates(weighted_titles, preferences, hyperparameters)
    scored = recs._score_candidates(
        weighted_titles,
        candidates,
        preferences,
        hyperparameters,
        feedback_actions=feedback_actions,
    )
    ranked = [candidate for candidate, score, signals in recs._diversify(scored, hyperparameters, limit=max(K_VALUES))]
    return ranked, {
        "candidate_count": len(candidates),
        "eligible_candidate_count": len(scored),
        "metadata_richness": hyperparameters["metadata_richness"],
        "profile_mode": hyperparameters["profile_mode"],
        "score_weights": hyperparameters["score_weights"],
        "feedback_counts": dict(Counter(feedback_actions.values())),
    }

## Profile Diagnostics

In [ ]:
def diagnostics(profile, train_events, holdout_events, weighted_titles, candidates):
    train_titles = [row["title"] for row in weighted_titles]
    missing_tmdb = sum(1 for title in train_titles if not title.tmdb_id)
    missing_genres = sum(1 for title in train_titles if not title.genres)
    missing_language = sum(1 for title in train_titles if not title.original_language)
    feedback_counts = Counter(
        RecommendationFeedback.objects.filter(profile=profile).values_list("action", flat=True)
    )
    return {
        "profile": profile.name,
        "train_events": len(train_events),
        "holdout_events": len(holdout_events),
        "train_titles": len({event.title_id for event in train_events}),
        "holdout_titles": len({event.title_id for event in holdout_events}),
        "missing_tmdb_titles": missing_tmdb,
        "missing_genre_titles": missing_genres,
        "missing_language_titles": missing_language,
        "candidate_count": len(candidates),
        "feedback_counts": dict(feedback_counts),
    }

## Run Benchmark

In [ ]:
metric_rows = []
diagnostic_rows = []
inspection_rows = []

for profile in NetflixProfile.objects.all().order_by("name"):
    train_events, holdout_events = profile_windows(profile)
    if not train_events or not holdout_events:
        continue
    period_end = max(event.started_at.date() for event in train_events)
    weighted_titles = weighted_rows_from_events(train_events, period_end)
    preferences = recs._profile_preferences(weighted_titles)
    eligible = eligible_catalog(weighted_titles)
    diagnostic_rows.append(diagnostics(profile, train_events, holdout_events, weighted_titles, eligible))

    for baseline_name, baseline_func in BASELINES.items():
        ranked = baseline_func(weighted_titles, preferences, limit=max(K_VALUES))
        for k in K_VALUES:
            metric_rows.append(
                metric_row(
                    profile.name,
                    baseline_name,
                    k,
                    ranked,
                    holdout_events,
                    extra={"candidate_count": len(eligible), "eligible_candidate_count": len(eligible)},
                )
            )

    for experiment in EXPERIMENTS:
        ranked, extra = production_ranked(profile, weighted_titles, preferences, experiment)
        for rank, candidate in enumerate(ranked[:10], start=1):
            inspection_rows.append({
                "profile": profile.name,
                "model": experiment["name"],
                "rank": rank,
                "title": candidate.title,
                "media_type": candidate.media_type,
                "genres": candidate.genres,
                "language": candidate.original_language,
                "countries": candidate.origin_countries,
                "source_counts": source_counts([candidate]),
                "popularity": candidate.popularity,
            })
        for k in K_VALUES:
            metric_rows.append(metric_row(profile.name, experiment["name"], k, ranked, holdout_events, extra=extra))

metrics_df = pd.DataFrame(metric_rows)
diagnostics_df = pd.DataFrame(diagnostic_rows)
inspection_df = pd.DataFrame(inspection_rows)

metrics_df.head()

## Summary Tables

In [ ]:
summary_df = (
    metrics_df
    .groupby(["model", "k"], dropna=False)
    .agg(
        profiles=("profile", "nunique"),
        hit_rate=("hit_rate", "mean"),
        recall=("recall", "mean"),
        genre_coverage=("genre_coverage", "mean"),
        language_overlap=("language_overlap", "mean"),
        country_overlap=("country_overlap", "mean"),
        diversity=("diversity", "mean"),
        novelty=("novelty", "mean"),
        candidate_count=("candidate_count", "mean"),
        eligible_candidate_count=("eligible_candidate_count", "mean"),
    )
    .round(4)
    .reset_index()
)
summary_df.sort_values(["k", "recall", "genre_coverage"], ascending=[True, False, False])

In [ ]:
profile_summary_df = (
    metrics_df
    .groupby(["profile", "model", "k"], dropna=False)
    .agg(
        hit_rate=("hit_rate", "mean"),
        recall=("recall", "mean"),
        genre_coverage=("genre_coverage", "mean"),
        language_overlap=("language_overlap", "mean"),
        country_overlap=("country_overlap", "mean"),
        diversity=("diversity", "mean"),
        novelty=("novelty", "mean"),
    )
    .round(4)
    .reset_index()
)
profile_summary_df.head(20)

In [ ]:
ranked_summary_df = summary_df.assign(
    metadata_fit=summary_df[["genre_coverage", "language_overlap", "country_overlap"]].mean(axis=1).round(4)
)

best_by_recall_df = (
    ranked_summary_df
    .sort_values(
        ["k", "recall", "hit_rate", "genre_coverage", "language_overlap"],
        ascending=[True, False, False, False, False],
    )
    .groupby("k", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_metadata_df = (
    ranked_summary_df
    .sort_values(
        ["k", "metadata_fit", "diversity", "novelty"],
        ascending=[True, False, False, False],
    )
    .groupby("k", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_by_recall_df[["k", "model", "hit_rate", "recall", "genre_coverage", "language_overlap", "country_overlap", "diversity", "novelty"]])
display(best_by_metadata_df[["k", "model", "metadata_fit", "genre_coverage", "language_overlap", "country_overlap", "diversity", "novelty"]])

## Diagnostics And Inspection

In [ ]:
diagnostics_df.sort_values(["candidate_count", "missing_tmdb_titles"], ascending=[True, False]).head(20)

In [ ]:
inspection_df.head(30)

## Visual Checks

In [ ]:
try:
    import plotly.express as px
    display(px.bar(summary_df, x="model", y="recall", color="model", facet_col="k", title="Average Recall by Model and K"))
    display(px.bar(summary_df, x="model", y="genre_coverage", color="model", facet_col="k", title="Average Genre Coverage by Model and K"))
    display(px.scatter(metrics_df, x="candidate_count", y="recall", color="model", facet_col="k", title="Candidate Count vs Recall"))
except Exception as exc:
    print(f"Plotly charts skipped: {exc}")

## Export Results

In [ ]:
timestamp = pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")
metrics_path = EXPORT_DIR / f"recommender_metrics_{timestamp}.csv"
summary_path = EXPORT_DIR / f"recommender_summary_{timestamp}.csv"
best_recall_path = EXPORT_DIR / f"recommender_best_by_recall_{timestamp}.csv"
best_metadata_path = EXPORT_DIR / f"recommender_best_by_metadata_{timestamp}.csv"
diagnostics_path = EXPORT_DIR / f"recommender_diagnostics_{timestamp}.csv"
inspection_path = EXPORT_DIR / f"recommender_inspection_{timestamp}.csv"
json_path = EXPORT_DIR / f"recommender_benchmark_{timestamp}.json"

metrics_df.to_csv(metrics_path, index=False)
summary_df.to_csv(summary_path, index=False)
best_by_recall_df.to_csv(best_recall_path, index=False)
best_by_metadata_df.to_csv(best_metadata_path, index=False)
diagnostics_df.to_csv(diagnostics_path, index=False)
inspection_df.to_csv(inspection_path, index=False)
json_path.write_text(
    json.dumps(
        {
            "metrics": metrics_df.to_dict("records"),
            "summary": summary_df.to_dict("records"),
            "best_by_recall": best_by_recall_df.to_dict("records"),
            "best_by_metadata": best_by_metadata_df.to_dict("records"),
            "diagnostics": diagnostics_df.to_dict("records"),
            "inspection": inspection_df.to_dict("records"),
        },
        default=str,
        indent=2,
    )
)

{
    "metrics": str(metrics_path),
    "summary": str(summary_path),
    "best_by_recall": str(best_recall_path),
    "best_by_metadata": str(best_metadata_path),
    "diagnostics": str(diagnostics_path),
    "inspection": str(inspection_path),
    "json": str(json_path),
}

## How To Use Results

Promote a recommender change only when it beats baselines across multiple profiles and `k` values. A good change should improve at least one of recall, genre coverage, language/country overlap, or diversity without causing candidate count or novelty to collapse. Start with `best_by_recall_df` and `best_by_metadata_df` before drilling into per-profile rows.

Use `diagnostics_df` when a profile performs poorly. Common causes are thin holdout data, missing TMDB IDs, empty metadata fields, or too few eligible cached candidates.